In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Executorの例

<Accordion>
<AccordionItem title="パッケージバージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

このセクションの例は、Executorプリミティブのいくつかの一般的な使い方を示しています。これらの例を実行する前に、[Qiskitのインストール](/guides/install-qiskit)と[Executorクイックスタート](/guides/directed-execution-model)の指示に従ってください。

## 始める前に
このページのコード例の一部は、Samplomaticパッケージの一部である`samplex`を使用しています。したがって、これらのコードブロックを実行する前に、次のコードブロックに示すようにSamplomaticをインストールする必要があります。詳細については、[Samplomaticドキュメント](https://qiskit.github.io/samplomatic)を参照してください。

In [1]:
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Generate the circuit
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)
circuit.h(2)
circuit.cz(1, 2)
circuit.h(2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)
circuit.rz(Parameter("lam"), 2)
circuit.measure_all()

## 例：パラメータ化されたcircuit
この例は、パラメータを持つcircuitアイテムの追加方法と、samplexアイテムの追加方法を示しています。次のステップで構成されています：
1. circuitの設定：ターゲットcircuitを生成してトランスパイルする。
2. samplexの準備：ゲートと測定をアノテーション付きボックスにグループ化し、circuitテンプレートとsamplexのペアを生成する。
3. 実行：`QuantumProgram`にcircuitアイテムとsamplexアイテムを追加し、単一のジョブで両方を実行する。

### circuitの設定
3-qubit GHZ状態を準備し、qubitをPauli-Z軸の周りに回転させ、計算基底でqubitを測定します。

In [2]:
# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Transpile the circuit to ISA
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)
isa_circuit = preset_pass_manager.run(circuit)

バックエンドを指定し、QPUがサポートする命令のみを使用するようにcircuitをトランスパイルします（命令セットアーキテクチャ（ISA）circuitと呼ばれます）。

In [3]:
boxing_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxing_pm.run(isa_circuit)

### samplexの準備
`generate_boxing_pass_manager`便利関数とそのtwirlingパラメータを使用して、2-qubitゲートと測定をボックスにグループ化し、twirlingアノテーションを適用します。

In [4]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

`build`メソッドを使用して、テンプレートcircuitとsamplexを生成します。

In [5]:
# Generate a quantum program
program = QuantumProgram(shots=1024)

### circuitの実行
ExecutorはQuantumProgramオブジェクトを実行します。各`QuantumProgram`にはいくつかのアイテムを含めることができます。この例では、実行のためにcircuitアイテムとsamplexアイテムを追加します。詳細については、[Executorの入力と出力](/guides/executor-input-output)を参照してください。

最初のステップは、空のプログラムを初期化し、各アイテムの各設定に対して`1024`ショットを要求することです。

In [6]:
# Append the circuit and the parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(10, 3),  # 10 sets of parameter values
)

`QuantumProgram`にcircuitアイテムを追加します。このcircuitアイテムは、ISA circuitと10セットのパラメータ値の2つの部分で構成されています。

In [7]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        "parameter_values": np.random.rand(
            10, 3
        ),  # 10 sets of parameter values
    },
    shape=(2, 14, 10),
)

次の引数でsamplexアイテムを`QuantumProgram`に追加します：
- `build`関数によって生成されたテンプレートcircuitとsamplex
- 元のcircuitの10セットのパラメータ値
- 実行するランダム化の数

In [8]:
# initialize an Executor with default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

### Executorジョブの実行

In [9]:
# Access the results of the classical register of task #0, the CircuitItem
result_0 = result[0]["meas"]

# Access the results of the classical register of task #1, the SamplexItem
result_1 = result[1]["meas"]

各タスクの結果を取得します。

In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Prepare a circuit

num_qubits = 10
num_layers = 10

qubits = list(range(num_qubits))
circuit = QuantumCircuit(num_qubits)

for layer_idx in range(num_layers):
    circuit.rx(Parameter(f"theta_{layer_idx}"), qubits)
    for i in range(num_qubits // 2):
        circuit.cz(qubits[2 * i], qubits[2 * i + 1])

    circuit.rx(Parameter(f"phi_{layer_idx}"), qubits)
    for i in range(num_qubits // 2 - 1):
        circuit.cz(qubits[2 * i] + 1, qubits[2 * i + 1] + 1)

circuit.draw("mpl", scale=0.35, fold=100)

<Image src="../docs/images/guides/executor-examples/extracted-outputs/f9e93b2c-154a-4d09-872d-f770bcc669c4-0.svg" alt="Output of the previous code cell" />

## 例：PECの実行
この例は、samplexアイテムを使用して確率的エラーキャンセル（[PEC](/guides/error-mitigation-and-suppression-techniques#pec)）によるエラー軽減を実行する方法を示しています。

10個のqubitと2つのCXゲートのユニークなレイヤーを持つcircuitのミラーバージョンを考えてみましょう。これらが主なタスクです：
- twirlingを適用してcircuitを実行する。
- 論文["Probabilistic error cancellation with sparse Pauli-Lindblad models on noisy quantum processors"](https://arxiv.org/abs/2201.09866)のようにPEC軽減を適用してcircuitを実行する。

パイプラインは次のステップで構成されています：
1. 設定：ターゲットcircuitを生成し、その操作をボックスにグループ化する。
2. 学習：PECで軽減したい命令のノイズを学習する。
3. 実行：バックエンドでcircuitを実行する。
4. 分析：結果を後処理して分析する。

比較のために、このミラーcircuitを2回実行します。1回目はPauli-twirlingのみ適用し、2回目はPEC軽減を適用します。

> **Note:** この例の使用量は、Heron r2プロセッサで約10分です。

### circuitの設定
バックエンドを選択し、10-qubit circuitを準備します。

In [11]:
mirror_circuit = circuit.compose(circuit.inverse())
mirror_circuit.measure_all()

mirror_circuit.draw("mpl", scale=0.35, fold=100)

<Image src="../docs/images/guides/executor-examples/extracted-outputs/f8ac3f75-88ca-40f9-8382-6a427303bb8e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/executor-examples/extracted-outputs/f9e93b2c-154a-4d09-872d-f770bcc669c4-0.svg)

circuitとその逆を結合してミラーcircuitを作成します。

In [12]:
import numpy as np

parameter_values = np.random.rand(mirror_circuit.num_parameters)

![Output of the previous code cell](../docs/images/guides/executor-examples/extracted-outputs/f8ac3f75-88ca-40f9-8382-6a427303bb8e-0.svg)

いくつかのパラメータ値を設定します：

In [13]:
preset_pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3,
)

isa_circuit = preset_pass_manager.run(mirror_circuit)

パスマネージャーを使用して、circuitをISA circuitにトランスパイルします。

In [14]:
# Pass manager used to create twirled-annotated boxes.
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)

mirror_circuit_twirl = boxing_pm.run(isa_circuit)

# Pass manager used to create a new boxed circuit with
# both Twirl and InjectNoise annotations.
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)

mirror_circuit_pec = boxing_pm.run(isa_circuit)

次に、ゲートと測定をアノテーション付きボックスにグループ化します。これは手動で行うか、`generate_boxing_pass_manager`関数を使用して便利に行うことができます。最初のcircuitにはtwirlingのみが適用されるため、`Twirl`アノテーションのみが必要です。2番目のcircuitはPEC軽減で実行され、`Twirl`と`InjectNoise`アノテーションの両方が必要です。

In [15]:
from samplomatic.utils import find_unique_box_instructions

unique_box_instructions = find_unique_box_instructions(
    mirror_circuit_pec.data
)
assert len(unique_box_instructions) == 3

### ノイズの学習
ノイズ学習実験の数を最小限に抑えるために、2番目のcircuit（`InjectNoise`でアノテーション付きのボックスを持つもの）のユニークな命令を特定します。ユニーク性を定義する際、2つのボックス命令は次の両方が真の場合に等しいとみなされます：
- 内容が1-qubitゲートまで等しい。
- `Twirl`アノテーションが等しい（他のすべてのアノテーションは無視される）。

これにより、奇数と偶数のゲートボックス、および最終測定ボックスの3つのユニークな命令が生まれます。

In [16]:
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

learner = NoiseLearnerV3(backend)

learner.options.shots_per_randomization = 128
learner.options.num_randomizations = 32
learner.options.layer_pair_depths = [0, 1, 2, 4, 16, 32]

learner_job = learner.run(unique_box_instructions)

learner_job.job_id()
learner_result = learner_job.result()

`NoiseLearnerV3`を初期化し、そのオプションを設定して学習パラメータを選択し、ノイズ学習ジョブを実行します。

In [17]:
noise_maps = learner_result.to_dict(
    instructions=unique_box_instructions, require_refs=False
)

`result.to_dict`メソッドを使用して、`result`をsamplexが必要とするオブジェクトに変換します。

In [18]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram

# Initialize an empty QuantumProgram
program = QuantumProgram(shots=1000)

### circuitの実行
`Executor`はQuantumProgramオブジェクトを実行します。各`QuantumProgram`にはいくつかの*アイテム*を含めることができ、プログラムに追加されます。各アイテムはプログラムが実行するタスクです。

空のプログラムを初期化し、各アイテムの各設定に対して`1000`ショットを要求します。

In [19]:
template_twirl, samplex_twirl = build(mirror_circuit_twirl)

program.append_samplex_item(
    template_twirl,
    samplex=samplex_twirl,
    samplex_arguments={"parameter_values": parameter_values},
    shape=(900,),
)

次に、`mirror_circuit_twirl`のテンプレートcircuitとsamplexをビルドし、プログラムに追加します。また、samplexから`900`ランダム化を要求します。これは、samplexが`900`セットのパラメータを生成し、各セットがQPUで`1000`回（shots数）実行されることを意味します。

これはプログラムの最初のタスク（結果0）です。

In [20]:
template_pec, samplex_pec = build(mirror_circuit_pec)

program.append_samplex_item(
    template_pec,
    samplex=samplex_pec,
    samplex_arguments={
        "parameter_values": parameter_values,
        "pauli_lindblad_maps": noise_maps,
        "noise_scales": {
            ref: -1.0 for ref in noise_maps
        },  # Set the scales to -1 for PEC
    },
    shape=(900,),
)

同様に、`mirror_circuit_pec`のテンプレートcircuitとsamplexを追加し、`900`ランダム化を要求します。これはプログラムの2番目のタスク（結果1）です。

In [21]:
from qiskit_ibm_runtime.executor import Executor

executor = Executor(backend)
executor_job = executor.run(program)

executor_job.job_id()

executor_results = executor_job.result()
executor_results

twirl_result = executor_results[0]

print(f"Twirl result keys:\n {list(twirl_result.keys())}\n")
print(f"Shape of results: {twirl_result['meas'].shape}")

pec_result = executor_results[1]

print(f"PEC result keys:\n {list(pec_result.keys())}\n")
print(f"Shape of results: {pec_result['meas'].shape}")

Twirl result keys:
 ['meas', 'measurement_flips.meas']

Shape of results: (900, 1000, 10)
PEC result keys:
 ['meas', 'measurement_flips.meas', 'pauli_signs']

Shape of results: (900, 1000, 10)


`Executor`をインポートしてジョブを送信します。

In [22]:
# Undo measurement twirling
twirl_result_unflipped = (
    twirl_result["meas"] ^ twirl_result["measurement_flips.meas"]
)

# Calculate the expectation values of single-qubit Z operators
exp_vals = 1 - 2 * twirl_result_unflipped.mean(axis=1).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 0.77
Qubit 1 -> 0.76
Qubit 2 -> 0.66
Qubit 3 -> 0.71
Qubit 4 -> 0.69
Qubit 5 -> 0.67
Qubit 6 -> 0.62
Qubit 7 -> 0.59
Qubit 8 -> 0.62
Qubit 9 -> 0.68


In [23]:
# Undo measurement twirling
pec_result_unflipped = (
    pec_result["meas"] ^ pec_result["measurement_flips.meas"]
)

# Calculate the signs for PEC mitigation
signs = np.prod((-1) ** pec_result["pauli_signs"], axis=-1)
signs = signs.reshape((signs.shape[0], 1))

# Calculate the expectation values of single-qubit Z operators as required by
# PEC mitigation
exp_vals = 1 - (2 * pec_result_unflipped.mean(axis=1) * signs).mean(axis=0)

for qubit, val in enumerate(exp_vals):
    print(f"Qubit {qubit} -> {np.round(val, 2)}")

Qubit 0 -> 0.98
Qubit 1 -> 0.99
Qubit 2 -> 0.96
Qubit 3 -> 0.98
Qubit 4 -> 0.98
Qubit 5 -> 0.98
Qubit 6 -> 0.98
Qubit 7 -> 0.95
Qubit 8 -> 0.95
Qubit 9 -> 0.94


### 結果の分析
最後に、10個のアクティブなqubitそれぞれに作用する1-qubit Pauli-Z演算子の期待値（期待値：`1.0`）を推定するために結果を後処理します。